# Air-routes analysis with GraphForge Python

## Goal

Use the same `data/air-routes/airports.csv` and `routes.csv` files as the GraphForge extension quickstart, but follow the workflow an analyst expects in a VS Code Jupyter notebook: inspect tabular data with pandas, build the graph through the native Python binding, run PageRank, visualize the result with Plotly, and save reusable project artifacts.

This is a parallel path, not a fallback engine. Graph operations still run in GraphForge's Rust core through the Python binding.

## Setup

Select a Python kernel that can import `graphforge`, `pyarrow`, `pandas`, and `plotly`. GraphForge's extension setup remains `uv`-only. If the selected kernel is missing packages, use its exact interpreter path in a VS Code terminal:

```text
uv pip install --python /path/to/python graphforge pyarrow pandas plotly ipykernel
```

The notebook does not install packages or choose a kernel on your behalf.

In [ ]:
from importlib.util import find_spec
from pathlib import Path
import json
import sys

required_packages = {
    'graphforge': 'graphforge',
    'pyarrow': 'pyarrow',
    'pandas': 'pandas',
    'plotly': 'plotly',
}
missing = [package for module, package in required_packages.items() if find_spec(module) is None]
if missing:
    package_list = ' '.join(missing)
    raise ModuleNotFoundError(
        f"Missing {package_list}. In a VS Code terminal run: "
        f"uv pip install --python {sys.executable} {package_list}"
    )

import graphforge as gf
import pandas as pd
import plotly.express as px

print(f'Python: {sys.executable}')
print(f'GraphForge: {gf.__version__}')

## Steps

### 1. Load and validate the shared dataset

The project marker anchors all paths. No data is downloaded and no hidden extension state is read.

In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        marker = candidate / 'FORMAT'
        if marker.exists() and marker.read_text(encoding='utf-8').strip() == 'graphforge-project/v1':
            return candidate
    raise FileNotFoundError('Open this notebook from the GraphForge air-routes sample project.')

project_root = find_project_root(Path.cwd())
data_root = project_root / 'data' / 'air-routes'
airports_path = data_root / 'airports.csv'
routes_path = data_root / 'routes.csv'

airports = pd.read_csv(airports_path)
routes = pd.read_csv(routes_path)
assert len(airports) == 586, f'Expected 586 airports, found {len(airports)}'
assert len(routes) == 7_430, f'Expected 7,430 routes, found {len(routes)}'
assert set(routes['from']).union(routes['to']).issubset(set(airports['id']))

display(airports.head(5))
print(f'{len(airports):,} airports · {len(routes):,} directed routes')

### 2. Build the same graph through the Python binding

Bulk publication keeps the notebook concise and returns Arrow receipts in logical input order. The receipts provide the native UUID endpoints used by the route batch. The graph is intentionally in-memory so rerunning the notebook starts cleanly.

In [ ]:
forge = gf.GraphForge()
airport_receipt = forge.add_nodes(
    'Airport',
    airports.rename(columns={'id': 'source_id'}),
    operation_uuid='018f0f4e-7b8c-7000-8000-00000000a001',
)
airport_uuid_by_id = dict(zip(
    airports['id'],
    airport_receipt.column('entity_uuid').to_pylist(),
))
route_batch = (
    routes.assign(
        src_id=routes['from'].map(airport_uuid_by_id),
        dst_id=routes['to'].map(airport_uuid_by_id),
    )
    .drop(columns=['from', 'to'])
)
route_receipt = forge.add_edges(
    'ROUTE',
    route_batch,
    operation_uuid='018f0f4e-7b8c-7000-8000-00000000a002',
)

assert airport_receipt.num_rows == len(airports)
assert route_receipt.num_rows == len(routes)
print(f'GraphForge graph: {forge.node_count("Airport"):,} Airport nodes · {route_receipt.num_rows:,} ROUTE edges')

### 3. Rank airports with Rust-owned PageRank

`rank()` returns a PyArrow table. Converting to pandas is an analyst-side presentation step; PageRank itself is not reimplemented in the notebook.

In [ ]:
ranked_airports = (
    forge.rank('Airport', by='pagerank', via='ROUTE', directed=True)
    .to_pandas()
    .sort_values(['score', 'code'], ascending=[False, True])
    .reset_index(drop=True)
)
ranked_airports['nodeUuid'] = ranked_airports['node_uuid'].map(bytes.hex)
ranked_airports['rank'] = ranked_airports.index + 1
analysis_columns = ['rank', 'code', 'city', 'region', 'score', 'lat', 'lon', 'nodeUuid']
top_airports = ranked_airports.loc[:, analysis_columns].head(25).copy()

assert len(ranked_airports) == len(airports)
assert ranked_airports['score'].is_monotonic_decreasing
display(top_airports.head(15).round({'score': 6}))

### 4. Save normal GraphForge project artifacts

The notebook publishes a bounded result and a complete Plotly visualization specification. After running this cell, use **GraphForge: Refresh Explorer**; the result and visualization can then be opened by the extension like any other saved project work.

In [ ]:
results_dir = project_root / 'results'
visualizations_dir = project_root / 'visualizations'
results_dir.mkdir(exist_ok=True)
visualizations_dir.mkdir(exist_ok=True)

result_path = results_dir / 'python-airport-pagerank.json'
result_rows = json.loads(top_airports.to_json(orient='records'))
result_payload = {
    'columns': analysis_columns,
    'rows': result_rows,
    'rowCount': len(result_rows),
}
result_path.write_text(json.dumps(result_payload, indent=2) + '\n', encoding='utf-8')

visualization_path = visualizations_dir / 'python-airport-pagerank.gfviz.json'
visualization_payload = {
    'format': 'graphforge.visualization/v2',
    'name': 'Airport PageRank — Python notebook',
    'kind': 'chart',
    'result': 'results/python-airport-pagerank.json',
    'filters': [],
    'renderer': {'id': 'plotly'},
    'chart': {
        'mark': 'bar',
        'bindings': {
            'x': 'code', 'y': 'score', 'color': 'region',
            'size': None, 'shape': None, 'series': None,
        },
        'aggregation': 'none',
        'binning': {'enabled': False, 'thresholds': None},
        'sort': [{'field': 'score', 'direction': 'descending'}],
        'axes': {'x': True, 'y': True},
        'legend': True,
        'theme': 'editor',
        'animation': False,
        'title': 'Top US airports by PageRank',
    },
}
visualization_path.write_text(
    json.dumps(visualization_payload, indent=2) + '\n',
    encoding='utf-8',
)
print(result_path.relative_to(project_root))
print(visualization_path.relative_to(project_root))

### 5. Visualize in the notebook

Plotly is the default analytical chart path in GraphForge and also works naturally in Python. The HTML export is self-contained and sits beside the saved GraphForge visualization specification.

In [ ]:
figure = px.bar(
    top_airports.head(15),
    x='code',
    y='score',
    color='region',
    hover_data=['city', 'rank'],
    title='Top US airports by PageRank',
    labels={'code': 'Airport', 'score': 'PageRank', 'region': 'Region'},
)
figure.update_layout(template='plotly_dark', xaxis_categoryorder='array', xaxis_categoryarray=top_airports.head(15)['code'])
figure.show()

html_path = visualizations_dir / 'python-airport-pagerank.html'
figure.write_html(html_path, include_plotlyjs=True, full_html=True)
print(html_path.relative_to(project_root))

## Checks

A successful top-to-bottom run proves all of the following from visible cells and files:

- the notebook read 586 airports and 7,430 directed routes from the same vendored CSVs as the extension;
- GraphForge Python bulk-created the graph and returned Arrow receipts;
- native PageRank returned one row per airport;
- the top 25 rows were written to `results/python-airport-pagerank.json`; and
- both the extension-readable Plotly spec and a self-contained HTML chart were written under `visualizations/`.

In [ ]:
assert result_path.exists()
assert visualization_path.exists()
assert html_path.exists()
print(top_airports.head(3)[['rank', 'code', 'score']].to_string(index=False))
forge.close()
print('Air-routes Python notebook complete.')

## Next Steps

Change the analytic (`pagerank` can be replaced by another supported `rank()` algorithm), reshape the Arrow result in pandas, or create another explicit project visualization. Keep source data under `data/`, tabular outputs under `results/`, and reproducible visualization choices under `visualizations/` so the extension, scripts, notebooks, and coding agents all see the same work.